# Una interfaz para un modelo

**Materiales desarrollados por Matías Barreto, 2026**

**Tecnicatura Superior en Ciencias de Datos e IA, IFTS N.º 24**
* **Nomenclatura oficial:** Procesamiento de Lenguaje Natural
* **Nombre de trabajo:** Laboratorio de PLN: Analítica, Textos y Cultura

---

| | |
|---|---|
| **Problema** | Un modelo dentro de un cuaderno solo lo puede usar quien sabe ejecutar código. |
| **Producto** | Una interfaz web local para un modelo, probada con casos difíciles, y una decisión sobre qué mostrar y qué advertir. |
| **Recursos** | `gradio` (ver más abajo), el modelo de sentimiento del cuaderno 01 (450 MB) y el de traducción (300 MB). CPU suficiente. |

Una oficina quiere que sus compañeros prueben un clasificador de sentimiento sin abrir un cuaderno. Con unas pocas líneas se puede darle una pantalla a cualquier función, incluida una que llama a un modelo de Hugging Face.

**¿Qué hace usable una interfaz y qué no puede arreglar?**

> **Extensión opcional.** No forma parte del recorrido obligatorio. Es un primer paso hacia el desarrollo de software con IA: vas a construir algo que otras personas puedan usar. Instalá Gradio junto con las librerías de la unidad, en un solo comando: `uv pip install -r requirements.txt -r extensiones/requirements-gradio.txt`.
>
> **Cómo se trabaja.** Primero explorás con el código ya escrito y las notas **Para mirar**. Después hay **microlaboratorios**: una tabla te dice qué líneas completar y la celda siguiente comprueba.

In [ ]:
import sys

import gradio as gr
from transformers import pipeline

# Este cuaderno vive en extensiones/: sumamos la carpeta de la unidad.
sys.path.insert(0, "..")
import utils

gr.__version__

## 1. Una función, una interfaz

**Gradio** es una biblioteca que convierte una función de Python en una página web. Una `Interface` une tres cosas: la función (`fn`), qué controles recibe (`inputs`) y cómo muestra el resultado (`outputs`). `launch()` arranca un servidor **en tu computadora**: solo vos podés abrir esa página. No usamos `share=True`, que crea un enlace público temporal hacia tu máquina.

In [ ]:
def saludar(nombre: str) -> str:
    """Arma un saludo.

    Args:
        nombre: Nombre que escribe la persona.

    Returns:
        El saludo.
    """
    return f"Hola, {nombre}."


demo_saludo = gr.Interface(
    fn=saludar, inputs="text", outputs="text", title="Saludo"
)
demo_saludo.launch()

> **Para mirar.** Escribí un nombre en la interfaz: la función es la misma que podrías llamar desde el cuaderno. La interfaz no agrega lógica, solo controles.

## 2. Una regla que parece un modelo

Antes de usar un modelo, una regla simple: si el texto dice «mal», es negativo; si dice «bien», es positivo. Se la puede poner detrás de la misma interfaz.

In [ ]:
def sentimiento_por_reglas(texto: str) -> str:
    """Clasifica con dos palabras clave.

    Args:
        texto: Frase a clasificar.

    Returns:
        Positivo, Negativo o Neutro.
    """
    minusculas = texto.lower()
    if "mal" in minusculas:
        return "Negativo"
    if "bien" in minusculas:
        return "Positivo"
    return "Neutro"


frases = [
    "Me fue muy bien en la entrevista.",
    "No estuvo mal la charla.",
    "La charla estuvo excelente.",
]
filas = []
for frase in frases:
    filas.append({"frase": frase, "regla": sentimiento_por_reglas(frase)})
utils.tabla(filas)

In [ ]:
demo_reglas = gr.Interface(
    fn=sentimiento_por_reglas,
    inputs="text",
    outputs="label",
    title="Sentimiento por reglas",
)
demo_reglas.launch()

> **Para mirar.**
>
> - La regla acierta con «Me fue muy bien…», pero se equivoca con «No estuvo mal la charla.» (la lee como negativa porque contiene `mal`) y con «excelente», que no está en su lista y sale `Neutro`.
> - Dentro de la interfaz falla igual: una interfaz hace usable una función, no la hace correcta.

## 3. Un modelo detrás de la interfaz

Reemplazamos la regla por el modelo del cuaderno 01. `top_k=None` pide el puntaje de las tres etiquetas, no solo el de la mejor, y `gr.Label` los muestra como barras. Si la persona deja el cuadro vacío, `gr.Error` le avisa en la pantalla en lugar de romper la aplicación.

El modelo se carga **una sola vez**, fuera de la función: cargarlo en cada consulta sería lentísimo.

In [ ]:
clasificador = pipeline(
    "text-classification",
    model="pysentimiento/robertuito-sentiment-analysis",
)
NOMBRES = {"POS": "Positivo", "NEG": "Negativo", "NEU": "Neutro"}


def puntajes_de(texto: str) -> dict:
    """Calcula el puntaje del modelo para cada etiqueta.

    Args:
        texto: Frase a clasificar.

    Returns:
        Diccionario con el nombre de cada etiqueta y su puntaje.
    """
    if not texto.strip():
        raise gr.Error("Escribi un texto para analizar.")
    puntajes = {}
    for resultado in clasificador(texto, top_k=None):
        puntajes[NOMBRES[resultado["label"]]] = resultado["score"]
    return puntajes


demo_modelo = gr.Interface(
    fn=puntajes_de,
    inputs=gr.Textbox(label="Texto", lines=3),
    outputs=gr.Label(num_top_classes=3, label="Sentimiento"),
    title="Sentimiento en español",
    examples=[
        ["La charla estuvo excelente, aprendí un montón."],
        ["Qué garrón de entrevista, no me fue nada bien."],
    ],
)
demo_modelo.launch()

> **Para mirar.** Probá los dos ejemplos y uno tuyo. La interfaz muestra tres barras que suman 1: se leen como una confianza, pero son el puntaje del modelo para cada etiqueta, no la probabilidad de que acierte.

### Microlaboratorio 1: Tu primera interfaz

**Tarea.** Ponerle una interfaz al traductor del cuaderno 01.  
**Producto.** `traducir` y `mi_interfaz`, comprobados.  
**Antes de empezar.** Secciones 1 y 3.

| Línea | Qué tiene que quedar | Con qué |
|---|---|---|
| (1) `return` | el texto traducido | `traductor(texto)[0]["translation_text"]` |
| (2) `mi_interfaz` | una interfaz que usa `traducir` | `gr.Interface` con `fn=traducir`, `inputs="text"` y `outputs="text"` |

**Comprobación.** `traducir` devuelve una traducción no vacía y `mi_interfaz` es una `Interface` que usa esa función.

> **Ayuda 1.** Mirá cómo `saludar` y `demo_saludo` se conectan en la sección 1.  
> **Ayuda 2.** El pipeline de traducción devuelve una lista; el texto está en la primera posición, en la clave `translation_text`.

In [ ]:
traductor = pipeline("translation", model="Helsinki-NLP/opus-mt-es-en")


def traducir(texto: str) -> str:
    """Traduce un texto del espanol al ingles.

    Args:
        texto: Texto en espanol.

    Returns:
        El texto en ingles.
    """
    return ""  # COMPLETAR (1): llamar a traductor y devolver el texto


mi_interfaz = None  # COMPLETAR (2): gr.Interface con fn, inputs y outputs
if mi_interfaz is not None:
    mi_interfaz.launch()

In [ ]:
utils.comprobar(
    traducir("Buenos días").strip() != "",
    "revisa (1): traducir tiene que devolver el texto traducido",
)
utils.comprobar(
    isinstance(mi_interfaz, gr.Interface) and mi_interfaz.fn is traducir,
    "revisa (2): mi_interfaz tiene que ser una gr.Interface con fn=traducir",
)

## 4. Probar con casos difíciles

Una interfaz invita a probar con lo primero que se te ocurre. Probamos a propósito lo que más cuesta: una ironía, una negación y un texto neutro. Llamamos a la función directamente para ver los tres puntajes.

In [ ]:
casos = [
    "Qué lindo que el colectivo llegue siempre tarde, ¿no?",
    "No estuvo mal la charla.",
    "La biblioteca abre de lunes a viernes.",
    "Me encantó la atención, volvería sin dudar.",
]
filas = []
for caso in casos:
    fila = {"texto": caso}
    fila.update(puntajes_de(caso))
    filas.append(fila)
utils.tabla(filas).round(3)

> **Para mirar.**
>
> - El modelo resuelve la negación que la regla no podía: «No estuvo mal la charla.» sale `Positivo` (`0.897`).
> - Con la ironía se equivoca, pero sin convicción: `Positivo` `0.390`, `Neutro` `0.336` y `Negativo` `0.275`. Tres barras casi parejas son una señal de duda que una interfaz que muestre solo la etiqueta ocultaría.
> - El aviso neutro sale `Neutro` (`0.829`).

## 5. Decidir qué mostrar y qué advertir

Dar una pantalla a un modelo cambia quién lo usa: ya no lo mira alguien que conoce sus límites. Antes de compartir la interfaz, decidí qué mostrarías (¿solo la etiqueta?, ¿los tres puntajes?) y qué advertencia pondrías en `description`.

**Tarea.** Escribí en `mi_decision` una oración condicional: para qué usos compartirías esta interfaz, para cuáles no, y qué mostrarías u ocultarías. Apoyate en la tabla de la sección 4.  

In [ ]:
mi_decision = ""  # COMPLETAR (3): tu decision, con su condicion

In [ ]:
utils.comprobar(
    len(mi_decision.split()) >= 15,
    "revisa (3): la decision tiene que tener al menos 15 palabras",
)

In [ ]:
# Apaga los servidores locales que abrimos en este cuaderno.
gr.close_all()

## Qué te llevás

- Una `Interface` une una función con controles de entrada y de salida; `launch()` la sirve solo en tu computadora.
- La interfaz hace usable una función, no la hace correcta: la regla y el modelo fallan con la negación y la ironía, también dentro de la pantalla.
- Cuando otras personas usan tu aplicación, qué mostrás y qué advertís es parte de la decisión.

**Sigue:** de un cuaderno a un Hugging Face Space, con `uv`, Git y un asistente de IA (cuaderno 06).